In [1]:
import pandas as pd

df = pd.read_csv('processed_stock_data.csv', encoding='utf-8')

In [2]:
df.head()

,Date,Price,Open,High,Low,Vol.,Change %,Symbol,return_day,return_week,...,WMA_50,WMA_100,EMA26,EMA12,MACD,Signal_Line,RSI_14,BB_upper,BB_lower,StochRSI_14
0,2010-01-25,17348.6,17348.6,17348.6,17348.6,7690.0,-4.86,CMG_RRV,-0.048050,-0.180182,...,13199.920941,14278.957287,17348.600000,17348.600000,0.000000,0.000000,0.0,16477.022982,12133.157018,0.938204
1,2010-01-26,16515.0,16515.0,16515.0,16515.0,47890.0,-4.80,CMG_RRV,-0.048050,-0.180182,...,13199.920941,14278.957287,17286.851852,17220.353846,-66.498006,-13.299601,0.0,16477.022982,12133.157018,0.938204
2,2010-01-27,15733.6,15733.6,15733.6,15733.6,11280.0,-4.73,CMG_RRV,-0.047315,-0.180182,...,13199.920941,14278.957287,17171.796159,16991.622485,-180.173674,-46.674416,0.0,16477.022982,12133.157018,0.938204
3,2010-01-28,14952.1,14952.1,14952.1,14952.1,3630.0,-4.97,CMG_RRV,-0.049671,-0.180182,...,13199.920941,14278.957287,17007.374221,16677.849795,-329.524426,-103.244418,0.0,16477.022982,12133.157018,0.938204
4,2010-01-29,14222.7,14222.7,14222.7,14222.7,44490.0,-4.88,CMG_RRV,-0.048782,-0.180182,...,13199.920941,14278.957287,16801.102057,16300.134442,-500.967615,-182.789057,0.0,16477.022982,12133.157018,0.938204


In [3]:
df.columns

Index(['Date', 'Price', 'Open', 'High', 'Low', 'Vol.', 'Change %', 'Symbol',
       'return_day', 'return_week', 'return_month', 'volatility_day',
       'volatility_week', 'volatility_month', 'liquidity_day',
       'liquidity_week', 'liquidity_month', 'z_score', 'SMA_20', 'SMA_50',
       'SMA_100', 'WMA_20', 'WMA_50', 'WMA_100', 'EMA26', 'EMA12', 'MACD',
       'Signal_Line', 'RSI_14', 'BB_upper', 'BB_lower', 'StochRSI_14'],
      dtype='object')

In [4]:
len(df)

71969

In [6]:
df = pd.read_csv("processed_stck_data/CMG_RRV.csv", parse_dates=["Date"])

In [ ]:
threshold = 0.005  # 0.5% (có thể điều chỉnh)

def classify_state(ret):
    if ret > threshold:
        return 1      # Tăng
    elif ret < -threshold:
        return -1     # Giảm
    else:
        return 0      # Đi ngang

df['state'] = df['return_day'].apply(classify_state)

In [8]:
df.sort_values(by=['Symbol', 'Date'], inplace=True)

In [9]:
states = [-1, 0, 1]  
transition_counts = pd.DataFrame(0, index=states, columns=states)

for symbol, group in df.groupby('Symbol'):
    group = group.sort_values(by='Date').reset_index(drop=True)
    for i in range(1, len(group)):
        prev_state = group.loc[i-1, 'state']
        curr_state = group.loc[i, 'state']
        transition_counts.loc[prev_state, curr_state] += 1
        
print("Ma trận đếm chuyển trạng thái:")
print(transition_counts)

Ma trận đếm chuyển trạng thái:
     -1    0    1
-1  507  312  566
 0  384  339  282
 1  494  354  518


In [10]:
# 4. Tính ma trận xác suất chuyển đổi
transition_prob = transition_counts.div(transition_counts.sum(axis=1), axis=0)
print("\nMa trận xác suất chuyển trạng thái:")
print(transition_prob)


Ma trận xác suất chuyển trạng thái:
          -1         0         1
-1  0.366065  0.225271  0.408664
 0  0.382090  0.337313  0.280597
 1  0.361640  0.259151  0.379209


In [11]:
# 5. Dự đoán trạng thái kế tiếp cho mỗi mã
predictions = {}
for symbol, group in df.groupby('Symbol'):
    last_state = group.sort_values(by='Date').iloc[-1]['state']
    prob_series = transition_prob.loc[last_state]
    predicted_state = prob_series.idxmax()
    predictions[symbol] = predicted_state

In [12]:
print("\nDự đoán trạng thái kế tiếp cho từng mã:")
for symbol, pred in predictions.items():
    state_str = "Tăng" if pred == 1 else "Giảm" if pred == -1 else "Đi ngang"
    print(f"{symbol}: {state_str}")


Dự đoán trạng thái kế tiếp cho từng mã:
CMG_RRV: Tăng
